<a href="https://colab.research.google.com/github/NatalieAleksandrova2026/DTA_2026/blob/main/ML/logreg_pipeline_TASKS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Воркбук: логістична регресія + Pipeline

Просте тренування на дві теми:
- **Логістична регресія** - класика класифікації, що дає ймовірності й інтерпретовні коефіцієнти.
- **Pipeline** - складаємо препроцесинг (масштабування + кодування) і модель в один надійний конвеєр.

**Набір даних:** клієнти сервісу (`clients`). Ціль - `upgraded` (1 = перейшов на преміум, 0 = ні).

| Стовпець | Що це | Тип |
|---|---|---|
| `age` | вік | число |
| `tenure` | місяців із сервісом | число |
| `usage` | годин/міс використання | число |
| `support` | звернень у підтримку | число |
| `plan` | тариф (базовий/стандарт/сімейний) | категорія |
| `region` | регіон | категорія |
| `upgraded` | перейшов на преміум - **ціль** | 0/1 |

**Як працювати:** запусти «Підготовку даних», іди по кроках, заповнюй `# TODO`. Підказки - під кожним кроком.


---

## 🔧 Підготовка даних (просто запусти)

In [5]:
# ▶️ Просто запусти цю комірку — вона готує дані. Міняти нічого не треба.
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)

# Задача: чи перейде клієнт на ПРЕМІУМ-підписку (1 = так, 0 = ні)
N = 900
age          = np.random.randint(18, 70, N)                       # вік
tenure       = np.random.randint(1, 60, N)                        # місяців із сервісом
usage        = np.random.normal(80, 35, N).clip(0, 220).round(0)  # годин/міс використання
support      = np.random.poisson(1.3, N)                          # звернень у підтримку

plan   = np.random.choice(["базовий", "стандарт", "сімейний"], N, p=[.45, .35, .20])
plan_bonus = pd.Series({"базовий": -0.4, "стандарт": 0.3, "сімейний": 1.1})

region = np.random.choice(["північ", "південь", "схід", "захід"], N)
region_bonus = pd.Series({"північ": 0.1, "південь": -0.1, "схід": 0.0, "захід": 0.2})

logit = (0.03*usage + 0.045*tenure - 0.35*support - 0.012*age
         + plan_bonus[plan].values + region_bonus[region].values
         - 3.0 + np.random.normal(0, 0.8, N))
upgraded = (logit > 0).astype(int)

clients = pd.DataFrame({
    "age": age, "tenure": tenure, "usage": usage.astype(int), "support": support,
    "plan": plan, "region": region, "upgraded": upgraded,
})

print("✅ Дані готові. Таблиця clients:", clients.shape)
print("Частка тих, хто перейшов на преміум:", f"{clients['upgraded'].mean():.0%}")

✅ Дані готові. Таблиця clients: (900, 7)
Частка тих, хто перейшов на преміум: 48%


In [6]:
# Подивись на дані
clients.head()

,age,tenure,usage,support,plan,region,upgraded
0,56,17,79,4,базовий,схід,0
1,69,5,61,2,базовий,південь,0
2,46,29,24,0,базовий,північ,0
3,32,4,100,0,стандарт,захід,1
4,60,10,52,0,стандарт,захід,0


---
### Крок 1. Розвідка: баланс класів і типи ознак
Виведи частку кожного класу в `upgraded` і визнач, які стовпці числові, а які категорійні.

*Підказка:* `clients["upgraded"].value_counts(normalize=True)`.

In [7]:
# TODO: виведи баланс класів
clients["upgraded"].value_counts(normalize=True)

,proportion
upgraded,
0,0.515556
1,0.484444


In [8]:
clients.dtypes

,0
age,int64
tenure,int64
usage,int64
support,int64
plan,object
region,object
upgraded,int64


✍️ Випиши списки стовпців (знадобляться далі):
> числові: age, tenure, usage,support, upgraded  
> категорійні: plan, region

### Крок 2. X, y і поділ train / test
- `X` — усі стовпці, КРІМ `upgraded`. `y` — `upgraded`.
- Поділ: 20% у тест, `random_state=RANDOM_STATE`, **`stratify=y`** (щоб пропорція класів збереглась).

*Підказка:* `train_test_split(X, y, test_size=.., random_state=.., stratify=..)`.

In [9]:
clients.columns

Index(['age', 'tenure', 'usage', 'support', 'plan', 'region', 'upgraded'], dtype='object')

In [10]:
from sklearn.model_selection import train_test_split

# TODO: створи X, y та зроби поділ
X = clients[['age', 'tenure', 'usage', 'support', 'plan', 'region']]
y = clients['upgraded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y

)

### Крок 3. Опиши, що робити з кожним типом стовпців (`ColumnTransformer`)
Числові — **масштабувати** (`StandardScaler`); категорійні — **One-Hot** (`OneHotEncoder`).
Логістичній регресії масштабування потрібне (у ній є регуляризація).

*Підказка:*
```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = [..]
cat_cols = [..]

preprocess = ColumnTransformer([
    ("num", .., num_cols),
    ("cat", .., cat_cols),
])
```

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# TODO: задай num_cols, cat_cols і збери preprocess

num_cols = ['age', 'tenure', 'usage', 'support']
cat_cols = ['plan', 'region']

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(), cat_cols)
])

### Крок 4. Збери повний `Pipeline`: препроцесинг + модель
Поклади `preprocess` і `LogisticRegression(max_iter=1000)` в один `Pipeline`.

*Підказка:*
```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("prep", ..),
    ("model", ..),
])
```

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# TODO: збери pipe
pipe = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=1000))
])




### Крок 5. Навчи конвеєр і виміряй accuracy на тесті
Один виклик `.fit(X_train, y_train)` прожене дані через усі кроки.

*Підказка:* `pipe.fit(...)`, далі `pipe.score(X_test, y_test)`.

In [13]:
# TODO: навчи pipe і виведи accuracy на тесті
pipe.fit(X_train, y_train)

print('Accuracy на тесте: ', round(pipe.score(X_test, y_test), 3))

Accuracy на тесте:  0.844


### Крок 6. Деталізована оцінка: матриця плутанини й звіт
Передбач класи на тесті, побудуй `confusion_matrix` і `classification_report`.

*Підказка:* `pipe.predict(X_test)`; `confusion_matrix(...)`; `classification_report(...)`.

In [14]:
from sklearn.metrics import confusion_matrix, classification_report
y_pred = pipe.predict(X_test)

# TODO: передбач, виведи матрицю плутанини та звіт


cm = pd.DataFrame(confusion_matrix(y_test, y_pred),
                  index=["Насправді: ні", "Насправді: так"],
                  columns=["Прогноз: ні", "Прогноз: так"])
classification_report(y_test, y_pred, target_names=["Насправді: ні", "Насправді: так"])

display(cm)

print(classification_report(y_test, y_pred, target_names=["Not upgraded", "Upgraded"]))




,Прогноз: ні,Прогноз: так
Насправді: ні,80,13
Насправді: так,15,72


              precision    recall  f1-score   support

Not upgraded       0.84      0.86      0.85        93
    Upgraded       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180



### Крок 7. Ймовірності + ROC-AUC
Логістична регресія дає не лише мітку, а й **ймовірність**. Дістань ймовірність класу «1» і порахуй ROC-AUC.

*Підказка:* `proba = pipe.predict_proba(X_test)[:, 1]`; `roc_auc_score(y_test, proba)`.

In [15]:
from sklearn.metrics import roc_auc_score

# TODO: дістань proba та порахуй ROC-AUC
proba = pipe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
print(f'ROC-AUC: {auc:.4f}')

pd.DataFrame({
    'predict': pipe.predict(X_test)[:5],
    'predict_proba': pipe.predict_proba(X_test)[:,1][:5]
})

ROC-AUC: 0.9267


,predict,predict_proba
0,0,0.067650
1,0,0.151676
2,0,0.001724
3,1,0.968892
4,0,0.187364


### Крок 8. 🔑 Інтерпретація коефіцієнтів
Дістань назви ознак після препроцесингу й коефіцієнти моделі. Знак: **+ підвищує** ймовірність переходу, − знижує.

*Підказка:*
```python
names = ..
coefs = ..
```
Зведи у `DataFrame` і відсортуй за модулем.

In [16]:
# TODO: побудуй таблицю "ознака — коефіцієнт", відсортовану за |коеф.|
feature_names = pipe.named_steps['prep'].get_feature_names_out()
coefs = pipe.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({
    'features': feature_names,
    'coef': coefs.round(4)
}).sort_values('coef', key=abs, ascending=False)

print('Знак: + підвищує ймовірність переходу, - знижує:')
coef_df

Знак: + підвищує ймовірність переходу, - знижує:


,features,coef
2,num__usage,2.0271
1,num__tenure,1.6484
6,cat__plan_сімейний,1.4120
4,cat__plan_базовий,-1.3397
3,num__support,-0.8359
7,cat__region_захід,0.5376
0,num__age,-0.4069
10,cat__region_схід,-0.3406
8,cat__region_південь,-0.1894
9,cat__region_північ,0.1362


✍️ **Відповідь словами:**
> Найсильніше підвищує шанс переходу на преміум ___, а знижує ___.    
**Strongest increase**  
**num__usage (+2.03):**  
the more actively a person uses the service, the higher the probability
they will upgrade to premium. This makes sense — users who engage
frequently already see the value of the product and are more willing
to pay for additional features.  
**Strongest decrease**:   
**cat__plan_базовий (-1.34):**
the basic plan itself does not motivate upgrading — customers are
satisfied with the minimum feature set and see no reason to pay more.

### Крок 9. Прогноз для нового клієнта
Конвеєр приймає **сирі** дані — кодувати/масштабувати вручну не треба. Створи клієнта й виведи і рішення, і ймовірність.

Клієнт: вік 30, tenure 24, usage 120, support 0, plan «сімейний», region «захід».

*Підказка:* `pd.DataFrame([{...}])` з тими самими назвами стовпців → `pipe.predict_proba(...)[0, 1]`.

In [17]:
# TODO: створи new_client, виведи рішення та ймовірність переходу
new_client = pd.DataFrame([{
    'age': 30,
    'tenure': 24,
    'usage': 120,
    'support': 0,

    'plan': 'сімейний',
    'region': 'захід'
}])

display(new_client)

pred = pipe.predict(new_client)[0]
proba = pipe.predict_proba(new_client)[0, 1]

print(f'Рішення:      {"перейде на преміум" if pred else "не перейде"}')


,age,tenure,usage,support,plan,region
0,30,24,120,0,сімейний,захід


Рішення:      перейде на преміум


### Крок 10. Чесна оцінка: крос-валідація всього конвеєра
Прожени `pipe` через `cross_val_score` (cv=5, scoring="roc_auc"). Бо весь препроцесинг усередині Pipeline — кожен фолд обробляється окремо, **без витоку**.

*Підказка:* `cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")`.

In [18]:
from sklearn.model_selection import cross_val_score

# TODO: крос-валідація, виведи середнє ± розкид
cv = cross_val_score(pipe, X, y, cv=5, scoring='roc_auc')
print(f'CV ROC_AUC: {np.round(cv, 3)} | average: {cv.mean():.3f} + {cv.std():.3f}')

CV ROC_AUC: [0.934 0.931 0.94  0.891 0.921] | average: 0.923 + 0.018


---
# ⭐ Бонус (необов'язково)
1. **Навіщо масштабування?** Збери другий конвеєр **без** `StandardScaler` (числові — `passthrough`) і порівняй ROC-AUC. Сильно змінилось?
```python
("num", "passthrough", num_cols)
```
2. **Дисбаланс класів.** Додай у `LogisticRegression(class_weight="balanced")` і подивись, як зміняться recall для класу «1» та матриця плутанини.
```python
LogisticRegression(max_iter=1000, class_weight="balanced")
```
3. **Поріг рішення.** Замість порогу 0.5 спробуй 0.3 (`proba >= 0.3`). Як зміняться precision і recall?
```python
# 3. Поріг 0.3 замість 0.5
import numpy as np
proba = pipe.predict_proba(X_test)[:, 1]
for thr in [0.5, 0.3]:
    pred_thr = (proba >= thr).astype(int)
    cm = confusion_matrix(y_test, pred_thr)
    print(f"Поріг {thr}: матриця\n{cm}")
print("→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)")
```

In [19]:
# Місце для бонусних експериментів
prep_scale = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)
])
pipe_scale = Pipeline([
    ('prep', prep_scale),
    ('model', LogisticRegression(max_iter=1000))
])

pipe_scale.fit(X_train, y_train)
proba_scale = pipe_scale.predict_proba(X_test)[:, 1]
auc_scale = roc_auc_score(y_test, proba_scale)

print(f'ROC-AUC зі scaling:    {auc:.4f}')
print(f'ROC-AUC без scaling:   {auc_scale:.4f}')
print(f'Різниця:               {abs(auc - auc_scale):.4f}')

ROC-AUC зі scaling:    0.9267
ROC-AUC без scaling:   0.9256
Різниця:               0.0011


In [23]:
prep_balanced = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)
])

pipe_balanced = Pipeline([
    ('prep', prep_balanced),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

pipe_balanced.fit(X_train, y_train)
y_pred_balanced = pipe_balanced.predict(X_test)

cm = pd.DataFrame(confusion_matrix(y_test, y_pred_balanced),
                  index=["Насправді: ні", "Насправді: так"],
                  columns=["Прогноз: ні", "Прогноз: так"])
classification_report(y_test, y_pred_balanced, target_names=["Насправді: ні", "Насправді: так"])

display(cm)

print(classification_report(y_test, y_pred_balanced, target_names=["Not upgraded", "Upgraded"]))

,Прогноз: ні,Прогноз: так
Насправді: ні,81,12
Насправді: так,15,72


              precision    recall  f1-score   support

Not upgraded       0.84      0.87      0.86        93
    Upgraded       0.86      0.83      0.84        87

    accuracy                           0.85       180
   macro avg       0.85      0.85      0.85       180
weighted avg       0.85      0.85      0.85       180



class_weight="balanced" had almost no impact—the recall for class "1" remained at 0.83. This means there is no class imbalance; the dataset is split roughly evenly between "churned" (87) and "retained" (93). Therefore, balanced had no effect—it is only needed when one class is much rarer, for example, 95% vs. 5%.

In [24]:
import numpy as np
proba = pipe.predict_proba(X_test)[:, 1]
for thr in [0.5, 0.3]:
    pred_thr = (proba >= thr).astype(int)
    cm = confusion_matrix(y_test, pred_thr)
    print(f"Поріг {thr}: матриця\n{cm}")
print("→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)")

Поріг 0.5: матриця
[[80 13]
 [15 72]]
Поріг 0.3: матриця
[[71 22]
 [ 8 79]]
→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)


---
# 🧠 Питання на розуміння (без коду)
1. Чому логістична регресія — це **класифікація**, попри слово «регресія» в назві?
2. Що показує `predict_proba` і чим воно корисніше за `predict` для бізнесу?
3. Навіщо взагалі загортати кроки в `Pipeline` — що поганого станеться, якщо масштабувати дані **до** `train_test_split`?
4. Логістичній регресії масштабування потрібне, а дереву рішень — ні. Чому?
5. Коефіцієнт `support` від'ємний. Як прочитати це вголос для керівника?

> 🎯 Якщо зібрав робочий Pipeline і впевнено читаєш коефіцієнти — ти володієш найбільш «продакшн-готовим» патерном класичного ML.

Чудові питання. Це якраз та база, на якій тримається розуміння роботи моделей у реальних бізнес-задачах. Давай розберемо все по поличках.

---

## 1. Чому логістична регресія — це класифікація, попри назву?

**Коротка відповідь:** Вона використовує регресію всередині себе, але її фінальний результат — це ймовірність належності до класу.

**Як це працює:** Логістична регресія спочатку рахує звичайне лінійне рівняння (регресію):


$$z = w_1x_1 + w_2x_2 + \dots + b$$


Значення $z$ може бути будь-яким: від мінус нескінченності до плюс нескінченності. Але нам для класифікації потрібна ймовірність (від 0 до 1).

Тому модель пропускає це число $z$ через спеціальну функцію — **сигмоїду (sigmoid function)**. Сигмоїда «стискає» будь-яке число у проміжок між 0 та 1.

> **Висновок:** Назва «регресія» залишилася через математичну основу (пошук лінійної залежності $z$), але оскільки вихід моделі перетворюється на ймовірність класу, алгоритм використовується виключно для **класифікації**.

---

## 2. Що показує `predict_proba` і чим воно корисніше за `predict` для бізнесу?

* `predict` видає жорстку мітку класу: `0` або `1` (наприклад, «клієнт залишиться» або «клієнт піде»). За замовчуванням межа (threshold) стоїть на рівні `0.5`.
* `predict_proba` видає **конкретну ймовірність** для кожного класу. Наприклад: `[0.12, 0.88]`. Це означає, що модель на 88% впевнена, що клієнт піде.

### Чому це золото для бізнесу?

1. **Пріоритезація ресурсів:** Якщо у вас 10 000 клієнтів «на відтік» за версією `predict`, у кол-центру не висить ресурсів обдзвонити всіх. Завдяки `predict_proba` ви можете відсортувати їх і зателефонувати спочатку тим, у кого ймовірність відтоку `> 0.95`.
2. **Гнучкість рішень (Гнучкі пороги):** Що якщо утримання клієнта коштує $5, а його втрата — $500? Тоді бізнесу вигідно реагувати навіть на тих клієнтів, у кого ймовірність відтоку всього `0.2`. З `predict` ви б їх пропустили, а з `predict_proba` ви можете змінити поріг класифікації з `0.5` на `0.2`.

---

## 3. Навіщо загортати кроки в Pipeline і що поганого в масштабуванні ДО `train_test_split`?

Якщо відмасштабувати (наприклад, за допомогою `StandardScaler`) весь датасет перед тим, як ділити його на train і test, станеться **витік даних (Data Leakage)**.

**Чому це погано:**
`StandardScaler` вираховує середнє значення ($\mu$) і стандартне відхилення ($\sigma$) по всьому датасету. Якщо ви зробите це до поділу, то інформація з майбутньої тестової вибірки (яку модель взагалі не повинна бачити) «просочиться» в тренувальну через ці загальні метрики.

В результаті:

* На валідації/тесті метрики будуть ідеальними (модель «підглядала» відповіді через масштаб).
* У продакшені на реальних даних модель покаже значно гірший результат.

**Навіщо потрібен Pipeline?**
Він гарантує, що `scaler.fit()` (обчислення середнього) відбудеться **тільки на train-даних**, а на тест-дані це середнє буде просто застосовано (`transform`). Pipeline автоматизує цей процес і захищає від випадкових помилок.

---

## 4. Чому логістичній регресії масштабування потрібне, а дереву рішень — ні?

### Логістична регресія (чутлива):

Вона шукає розділяючу площину і використовує градієнтний спуск (або інші оптимізатори) для пошуку коефіцієнтів (ваг $w$).

* Якщо одна ознака — це «Вік» (від 18 до 80), а друга — «Річний дохід» (від 10 000 до 5 000 000), то зміна доходу на одиницю майже не впливає, а зміна віку на одиницю — впливає сильно.
* Модель буде фокусуватися на великих числах (дохід), градієнтний спуск буде «іскрити» і збігатися дуже довго або взагалі застрягне.

### Дерево рішень (байдуже):

Дерево працює інакше — воно робить послідовні спліти (розбиття) за правилом: «Ознака X > Порогове значення?».

* Йому все одно, чи дохід вимірюється в доларах ($50,000), чи в тисячах доларів ($50). Правило «Дохід > 50 000» математично дасть точно такий самий поділ вибірки, як і «Дохід > 50». Масштаб кожної окремої ознаки не впливає на інші ознаки, бо дерево оцінює їх по черзі, а не разом у загальній формулі відстаней чи ваг.

---

## 5. Коефіцієнт support від'ємний. Як прочитати це вголос для керівника?

> ⚠️ **Важливе уточнення:** Метрика `support` у класичному звіті classification report (scikit-learn) — це просто **кількість реальних об'єктів** цього класу у вибірці. Вона **ніколи не може бути від'ємною** (не буває мінус 5 клієнтів).

Швидше за все, мова йде про **коефіцієнт (вагу / weight / coefficient)** логістичної регресії для якоїсь ознаки (наприклад, ознака називається `support` — техпідтримка).

### Як це прочитати для керівника, якщо коефіцієнт біля ознаки від'ємний:

Припустимо, ознака — це кількість звернень в техпідтримку (`support_calls`), а цільовий клас `1` — це «клієнт залишиться».

> **«Наш аналіз показує, що цей фактор має зворотний зв'язок із лояльністю клієнта. Простими словами: чим частіше клієнт звертається в підтримку (зростає показник support), тим нижча ймовірність того, що він залишиться з нами (знижується ймовірність класу 1). Кожне додаткове звернення суттєво підвищує ризик його відтоку».**

*(Якщо ж цільовий клас `1` — це «клієнт піде», а коефіцієнт від'ємний, то навпаки: «Чим більше звернень, тим менша ймовірність, що він піде — тобто підтримка працює чудово і задовільняє клієнта»).*